# Добор примеров редких классов через эмбеддинги внутри HDBSCAN-кластеров

Этот ноутбук делает семантический добор примеров для малочисленных классов.

Логика:

1. Загружаются уже размеченные данные.
2. Считается текущее количество примеров по классам.
3. Выбираются редкие классы, которым не хватает примеров.
4. Из уже размеченных примеров и описаний классов строятся эмбеддинги классов.
5. Кандидаты берутся из тех же HDBSCAN-кластеров, которые использовались в ноутбуке `extra_markup_from_hdbscan_clusters`.
6. Отзывы-кандидаты ранжируются по cosine similarity к редким классам.
7. Найденные отзывы проверяются той же LLM-разметкой через `qwen/qwen3-32b` и тот же prompt.
8. Сохраняются только отзывы, где LLM подтвердила хотя бы один целевой редкий класс.

## 1. Подключение Google Drive и установка библиотек

In [1]:
from google.colab import drive

drive.mount('/content/drive')

Mounted at /content/drive


In [2]:
!pip -q install groq pandas tqdm pyarrow openpyxl sentence-transformers scikit-learn

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 143.7/143.7 kB 3.5 MB/s eta 0:00:00


## 2. Импорты

In [3]:
import os
import re
import json
import time
import hashlib
from pathlib import Path
from getpass import getpass

import numpy as np
import pandas as pd
from tqdm.auto import tqdm
from groq import Groq
from sentence_transformers import SentenceTransformer

## 3. Пути и основные настройки

По умолчанию кандидаты берутся из тех же HDBSCAN-кластеров, которые выбирались через LLM-анализ кластеров.

Если захочешь искать вообще по всем строкам `hdbscan_clusters.parquet`, поставь:

```python
USE_ONLY_CANDIDATE_CLUSTERS_FROM_ANALYSIS = False
```

In [4]:
# =========================
# Пути проекта
# =========================
BASE_DIR = Path("/content/drive/MyDrive/MLops_project/project")
DATA_DIR = BASE_DIR / "data"

# Папка с результатами HDBSCAN и LLM-анализа кластеров
RESULT_DIR = DATA_DIR / "hdbscan_results_raw_sample_200k"

# Входные файлы из HDBSCAN / analyze_HDBSCAN
CLUSTERS_PATH = RESULT_DIR / "hdbscan_clusters.parquet"
CLUSTER_ANALYSIS_CSV_PATH = RESULT_DIR / "cluster_llm_analysis.csv"
ENRICHED_CLUSTERS_PATH = RESULT_DIR / "hdbscan_clusters_with_llm_analysis.parquet"

# Уже размеченные данные. Ноутбук использует все файлы, которые реально существуют.
EXISTING_LABELED_PATHS = [
    BASE_DIR / "data" / "labeled" / "wb_feedbacks_llm_labeled_from_sample" / "balanced_50_per_class_random_chunks.csv",
    BASE_DIR / "data" / "labeled" / "wb_feedbacks_llm_labeled_from_sample" / "balanced_50_per_class_random_chunks_final.csv",
    BASE_DIR / "data" / "labeled" / "wb_feedbacks_llm_labeled_from_clusters" / "extra_from_hdbscan_candidate_clusters.csv",
    BASE_DIR / "data" / "labeled" / "wb_feedbacks_llm_labeled_from_clusters" / "extra_from_hdbscan_candidate_clusters_final.csv",
]

# Куда сохранять новую разметку из embedding similarity
LABELED_DIR = BASE_DIR / "data" / "labeled" / "wb_feedbacks_llm_labeled_from_embeddings"
LABELED_DIR.mkdir(parents=True, exist_ok=True)

OUTPUT_BASENAME = "extra_from_embedding_similarity_rare_classes"
OUTPUT_CSV_PATH = LABELED_DIR / f"{OUTPUT_BASENAME}.csv"
OUTPUT_PARQUET_PATH = LABELED_DIR / f"{OUTPUT_BASENAME}.parquet"
FINAL_CSV_PATH = LABELED_DIR / f"{OUTPUT_BASENAME}_final.csv"
STATUS_CSV_PATH = LABELED_DIR / f"{OUTPUT_BASENAME}_class_status.csv"
RUN_STATS_JSON_PATH = LABELED_DIR / f"{OUTPUT_BASENAME}_run_stats.json"

# Кэш эмбеддингов, чтобы не пересчитывать при перезапуске
EMBEDDING_CACHE_DIR = LABELED_DIR / "embedding_cache"
EMBEDDING_CACHE_DIR.mkdir(parents=True, exist_ok=True)
CANDIDATE_EMBEDDINGS_NPY_PATH = EMBEDDING_CACHE_DIR / "candidate_embeddings.npy"
CANDIDATE_META_PARQUET_PATH = EMBEDDING_CACHE_DIR / "candidate_meta.parquet"
CLASS_EMBEDDINGS_NPY_PATH = EMBEDDING_CACHE_DIR / "class_embeddings.npy"
CLASS_EMBEDDINGS_JSON_PATH = EMBEDDING_CACHE_DIR / "class_embeddings_labels.json"
CLASS_PROTOTYPE_EMBEDDINGS_NPY_PATH = EMBEDDING_CACHE_DIR / "class_prototype_embeddings.npy"
CLASS_PROTOTYPE_META_CSV_PATH = EMBEDDING_CACHE_DIR / "class_prototype_meta.csv"

# =========================
# Колонки
# =========================
TEXT_COL = "text"
CLUSTER_COL = "cluster_id"
PROB_COL = "cluster_probability"

# =========================
# Режим выбора кандидатов
# =========================
# True = искать внутри тех же кластеров-кандидатов из cluster_llm_analysis.csv.
# False = искать по всем строкам hdbscan_clusters.parquet, кроме noise cluster -1.
USE_ONLY_CANDIDATE_CLUSTERS_FROM_ANALYSIS = True

# Если True, candidate clusters выбираются только по целевым редким классам.
# Если False, будут использоваться кластеры по всем ALLOWED_LABELS, как в старом ноутбуке.
SELECT_CLUSTERS_BY_TARGET_RARE_LABELS = True

MIN_CLUSTER_ANALYSIS_CONFIDENCE = 0.0
INCLUDE_NEGATIVE_OR_MIXED_WITHOUT_MATCHED_LABELS = False
MAX_CLUSTERS_TO_PROCESS = None
MAX_REVIEWS_PER_CLUSTER = None
SORT_REVIEWS_BY_CLUSTER_PROBABILITY = True

# =========================
# Редкие классы и лимиты
# =========================
TARGET_MIN_COUNT = 100

# Можно оставить None, тогда классы с count < TARGET_MIN_COUNT будут выбраны автоматически.
# Можно задать вручную список ниже.
MANUAL_TARGET_LABELS = [
    "Подделка / оригинальность",
    "Нейтральный / информационный отзыв",
    "Проблема с размером / посадкой",
    "Низкое качество материала",
    "Цена / ценность",
    "Другая проблема",
]

# Сколько ближайших кандидатов брать на каждый редкий класс до LLM-проверки.
TOP_K_PER_CLASS = 1600

# Ограничитель для тестового запуска. Для полного запуска оставь None.
MAX_REVIEWS_TO_SEND_TO_MODEL = None

# Сохранять отзыв только если LLM нашла хотя бы один целевой редкий класс.
SAVE_ONLY_IF_HAS_TARGET_LABEL = True

# Если True, как только все целевые классы достигли TARGET_MIN_COUNT, добор остановится.
STOP_WHEN_ALL_TARGETS_REACHED = True

# Дедупликация
DEDUP_BY_TEXT = True

# Если True, удалит старые выходные файлы этого ноутбука и начнет заново.
RESET_OUTPUT_FILES_ON_START = False
RESUME_FROM_EXISTING_OUTPUT = True

# =========================
# Эмбеддинги
# =========================
EMBEDDING_MODEL_NAME = "intfloat/multilingual-e5-base"
USE_E5_PREFIXES = True
EMBEDDING_BATCH_SIZE = 64

# Как смешивать центроид реальных примеров и описание класса.
# Для обычных классов можно использовать центроид уже размеченных отзывов.
# Для сложных / медленно собирающихся классов ниже включен расширенный поиск по нескольким прототипам.
EXAMPLES_CENTROID_WEIGHT = 0.70
DESCRIPTION_EMBEDDING_WEIGHT = 0.30
MIN_EXAMPLES_FOR_CENTROID = 3

# Новая логика: для каждого целевого класса строится не один эмбеддинг,
# а несколько query-прототипов: описание класса + типовые формулировки + часть реальных примеров.
# Близость кандидата к классу = максимум близости до прототипов этого класса.
# Это особенно помогает классам, которые плохо собирались одним усредненным центроидом.
USE_PROTOTYPE_QUERY_EXPANSION = True
USE_CLASS_DESCRIPTION_AS_PROTOTYPE = True
USE_EXISTING_EXAMPLES_AS_PROTOTYPES = True
MAX_EXAMPLES_AS_PROTOTYPES_PER_LABEL = 20
RANDOM_SEED_FOR_EXAMPLE_PROTOTYPES = 42

# Для этих классов один центроид часто слишком шумный или слишком общий.
# Поэтому им добавлены специальные поисковые прототипы ниже.
SLOW_LABELS_FOR_PROTOTYPE_SEARCH = [
    "Подделка / оригинальность",
    "Нейтральный / информационный отзыв",
    "Цена / ценность",
    "Другая проблема",
    "Проблема с размером / посадкой",
    "Низкое качество материала",
]

# Можно брать больше ближайших соседей для самых редких классов.
# Если класса нет в словаре, используется TOP_K_PER_CLASS.
TOP_K_BY_LABEL = {
    "Подделка / оригинальность": 2200,
    "Нейтральный / информационный отзыв": 2200,
    "Цена / ценность": 2000,
    "Другая проблема": 2000,
    "Проблема с размером / посадкой": 1800,
    "Низкое качество материала": 1800,
}

# =========================
# API и модель для финальной LLM-разметки
# =========================
API_KEY_ENV_NAME = "GROQ_API_KEY"
MODEL_NAME = "qwen/qwen3-32b"
TEMPERATURE = 0
MAX_RETRIES = 3
RETRY_BASE_SLEEP_SECONDS = 2
REQUEST_SLEEP_SECONDS = 0.0

print("CLUSTERS_PATH:", CLUSTERS_PATH)
print("CLUSTER_ANALYSIS_CSV_PATH:", CLUSTER_ANALYSIS_CSV_PATH)
print("OUTPUT_CSV_PATH:", OUTPUT_CSV_PATH)

CLUSTERS_PATH: /content/drive/MyDrive/MLops_project/project/data/hdbscan_results_raw_sample_200k/hdbscan_clusters.parquet
CLUSTER_ANALYSIS_CSV_PATH: /content/drive/MyDrive/MLops_project/project/data/hdbscan_results_raw_sample_200k/cluster_llm_analysis.csv
OUTPUT_CSV_PATH: /content/drive/MyDrive/MLops_project/project/data/labeled/wb_feedbacks_llm_labeled_from_embeddings/extra_from_embedding_similarity_rare_classes.csv


## 4. Разрешенные классы

In [5]:
ALLOWED_LABELS = [
    "Брак / дефект товара",
    "Низкое качество материала",
    "Проблема с размером / посадкой",
    "Несоответствие описанию",
    "Несоответствие ожиданиям / эффекту",
    "Проблема с комплектацией",
    "Проблема с упаковкой",
    "Проблема доставки / получения",
    "Цена / ценность",
    "Подделка / оригинальность",
    "Положительный отзыв",
    "Нейтральный / информационный отзыв",
    "Другая проблема",
]

PROBLEM_LABELS = [
    label for label in ALLOWED_LABELS
    if label not in ["Положительный отзыв", "Нейтральный / информационный отзыв", "Другая проблема"]
]

unknown_manual = [x for x in MANUAL_TARGET_LABELS if x not in ALLOWED_LABELS]
if unknown_manual:
    raise ValueError(f"В MANUAL_TARGET_LABELS есть неизвестные классы: {unknown_manual}")

print("Всего разрешенных классов:", len(ALLOWED_LABELS))

Всего разрешенных классов: 13


## 5. Prompt из старого ноутбука

Prompt оставлен таким же, чтобы новая разметка была совместима со старой.

In [6]:
USER_PROMPT = """
/no_think

Ты выполняешь разметку отзывов покупателей о товарах на маркетплейсе.

Твоя задача — определить, к каким классам относится отзыв. Это multi-label классификация: один отзыв может относиться к одному или нескольким классам.

Важно:

* Размечай только по тексту отзыва.
* Не додумывай факты, которых нет в отзыве.
* Не используй знания о товаре, бренде или маркетплейсе, если они не указаны в тексте.
* Выбирай класс только если в отзыве есть явный или достаточно понятный признак этого класса.
* Нельзя придумывать новые классы.
* Названия классов должны полностью совпадать со списком разрешенных классов.

Разрешенные классы:

1. "Брак / дефект товара"
   Выбирай, если товар сломан, поврежден, не работает, работает неправильно, быстро сломался, имеет трещины, сколы, заводской дефект или явную неисправность.
   Не выбирай этот класс, если товар просто сделан из плохого материала, но не сломан.

2. "Низкое качество материала"
   Выбирай, если покупатель жалуется на плохой материал, дешевый пластик, плохую ткань, кривые швы, хлипкость, слабую сборку, неприятный материал, быструю изнашиваемость.
   Не выбирай этот класс только из-за высокой цены или поломки.

3. "Проблема с размером / посадкой"
   Выбирай, если товар не подошел по размеру, форме, посадке или совместимости: маломерит, большемерит, неудобно сидит, не подошел к устройству, оказался меньше или больше ожидаемого.
   Если в отзыве явно сказано, что в карточке были неверные размеры или характеристики, можно дополнительно выбрать "Несоответствие описанию".

4. "Несоответствие описанию"
   Выбирай, если товар отличается от карточки товара, фото, описания, характеристик или заявленных свойств: другой цвет, другой материал, другая модель, не те характеристики, описание вводит в заблуждение.
   Не выбирай этот класс, если покупатель просто недоволен качеством, но не сравнивает товар с описанием.

5. "Несоответствие ожиданиям / эффекту"
   Выбирай, если товар формально пришел и не сломан, но не выполняет ожидаемую функцию или не дает результата: средство не помогает, устройство слабое, товар бесполезен, эффект отсутствует, результат хуже ожидаемого.
   Не выбирай этот класс, если товар именно неисправен или сломан.

6. "Проблема с комплектацией"
   Выбирай, если в заказе не хватает деталей, аксессуаров, инструкции, кабеля, насадок, креплений или других элементов комплекта; пришла только часть набора; комплект неполный.

7. "Проблема с упаковкой"
   Выбирай, если жалоба относится к упаковке: коробка мятая, упаковка вскрыта, порвана, плохая защита, товар был плохо упакован.
   Если поврежден только внешний вид упаковки, но товар целый, выбирай только этот класс.
   Если сам товар сломан или поврежден, дополнительно выбирай "Брак / дефект товара", если это явно следует из текста.

8. "Проблема доставки / получения"
   Выбирай, если проблема связана с доставкой или получением: долго ехал, задержали, перенесли доставку, пришел не туда, проблемы с пунктом выдачи, курьером, логистикой, заказ потеряли.
   Не выбирай этот класс только из-за мятой коробки — для этого есть "Проблема с упаковкой".

9. "Цена / ценность"
   Выбирай, если покупатель пишет, что товар не стоит своих денег, цена завышена, качество не соответствует цене, можно найти дешевле, за эти деньги ожидал большего.
   Не выбирай этот класс, если в отзыве просто плохое качество без упоминания цены или ценности.

10. "Подделка / оригинальность"
    Выбирай, если покупатель сомневается в подлинности товара или пишет, что товар похож на подделку, неоригинальный, не фирменный, отличается от оригинала, вызывает сомнения бренд, маркировка или упаковка.

11. "Положительный отзыв"
    Выбирай, если покупатель явно доволен товаром: товар понравился, все хорошо, рекомендую, качество устраивает, покупкой доволен, товар оправдал ожидания.
    Этот класс можно сочетать с проблемным классом, если отзыв смешанный: есть и похвала, и конкретная жалоба.
    Не выбирай этот класс, если в отзыве нет явной положительной оценки.

12. "Нейтральный / информационный отзыв"
    Выбирай, если отзыв не содержит явной положительной или отрицательной оценки, а только сообщает факт: получил, пока не использовал, дополню позже, купил в подарок, не открывал, пока ничего сказать не могу.
    Этот класс нельзя сочетать с другими классами.
    Не выбирай этот класс, если есть явная похвала или жалоба.

13. "Другая проблема"
    Выбирай, если в отзыве есть негативная проблема, но ее нельзя уверенно отнести ни к одному из основных классов.
    Этот класс используй только как запасной вариант для непонятной негативной жалобы.
    Не выбирай "Другая проблема", если можно выбрать более конкретный класс.
    Обычно этот класс не нужно сочетать с другими проблемными классами.

Правила выбора классов:

1. Один отзыв может иметь несколько классов.
2. Если отзыв содержит и похвалу, и жалобу, выбери "Положительный отзыв" и соответствующий проблемный класс.
3. "Нейтральный / информационный отзыв" выбирай только тогда, когда нет ни похвалы, ни жалобы.
4. "Нейтральный / информационный отзыв" не должен идти вместе с другими классами.
5. "Другая проблема" выбирай только если есть негатив, но нет подходящего конкретного класса.
6. Если проблема относится к самому товару, выбирай товарный класс: брак, качество, размер, комплектация, описание, эффект, цена, оригинальность.
7. Если проблема относится к процессу получения заказа, выбирай доставку или упаковку.
8. Не добавляй слишком много классов. Каждый выбранный класс должен подтверждаться фрагментом отзыва.
9. При сомнении между конкретным классом и "Другая проблема" выбирай конкретный класс.
10. При сомнении между "Положительный отзыв" и "Нейтральный / информационный отзыв" выбирай "Положительный отзыв" только при явной похвале.
11. Если отзыв очень короткий, например "норм", "ок", "пришло", "получил", и нет понятной оценки, выбирай "Нейтральный / информационный отзыв".
12. Если отзыв короткий, но содержит явную оценку, например "отлично", "супер", "ужас", классифицируй по этой оценке.

Как выставлять confidence:

* 0.90–1.00: класс очевиден, есть прямые слова из отзыва.
* 0.70–0.89: класс достаточно понятен, но формулировка не полностью прямая.
* 0.50–0.69: отзыв неоднозначный, но выбранный класс наиболее вероятен.
* ниже 0.50: используй только если отзыв очень неясный.

Формат ответа:

Верни только валидный JSON-объект без markdown и без текста вне JSON.

JSON должен иметь ровно такие поля:
{
"labels": ["один или несколько классов из разрешенного списка"],
"confidence": 0.0,
"evidence": "короткое объяснение на русском, какие слова из отзыва подтверждают выбор"
}
""".strip()

## 6. Подключение Groq API

In [7]:
if API_KEY_ENV_NAME not in os.environ or not os.environ[API_KEY_ENV_NAME]:
    os.environ[API_KEY_ENV_NAME] = getpass(f"Вставь {API_KEY_ENV_NAME}: ")

client = Groq(api_key=os.environ[API_KEY_ENV_NAME])

print("MODEL_NAME:", MODEL_NAME)
print("OUTPUT_CSV_PATH:", OUTPUT_CSV_PATH)

Вставь GROQ_API_KEY: ··········
MODEL_NAME: qwen/qwen3-32b
OUTPUT_CSV_PATH: /content/drive/MyDrive/MLops_project/project/data/labeled/wb_feedbacks_llm_labeled_from_embeddings/extra_from_embedding_similarity_rare_classes.csv


## 7. Вспомогательные функции для labels, hash и подсчета классов

In [8]:
def normalize_text_for_hash(text):
    text = str(text).strip().lower()
    text = " ".join(text.split())
    return text


def text_hash(text):
    return hashlib.md5(normalize_text_for_hash(text).encode("utf-8")).hexdigest()


def parse_labels_str(value):
    if isinstance(value, list):
        raw = value
    elif pd.isna(value):
        raw = []
    else:
        text = str(value).strip()
        if not text:
            raw = []
        else:
            try:
                parsed = json.loads(text)
                raw = parsed if isinstance(parsed, list) else [text]
            except Exception:
                raw = re.split(r"\s*;\s*|\s*,\s*", text)

    labels = []
    for item in raw:
        label = str(item).strip().strip('"').strip("'")
        if label in ALLOWED_LABELS and label not in labels:
            labels.append(label)
    return labels


def find_text_col(df):
    candidates = [TEXT_COL, "отзыв", "review", "review_text", "text"]
    for col in candidates:
        if col in df.columns:
            return col
    raise ValueError(f"Не найдена текстовая колонка. Колонки: {df.columns.tolist()}")


def find_labels_col(df):
    candidates = ["labels", "labels_str", "labels_counted_str", "matched_labels_str"]
    for col in candidates:
        if col in df.columns:
            return col
    raise ValueError(f"Не найдена колонка labels. Колонки: {df.columns.tolist()}")


def count_labels_from_rows(rows):
    counts = {label: 0 for label in ALLOWED_LABELS}
    for row in rows:
        labels = row.get("labels_counted", row.get("labels", []))
        if isinstance(labels, str):
            labels = parse_labels_str(labels)
        for label in labels:
            if label in counts:
                counts[label] += 1
    return counts


def count_labels_df(df, labels_col="labels"):
    counts = {label: 0 for label in ALLOWED_LABELS}
    for labels in df[labels_col].apply(parse_labels_str):
        for label in labels:
            if label in counts:
                counts[label] += 1
    return counts


def normalize_vector(x):
    x = np.asarray(x, dtype="float32")
    norm = np.linalg.norm(x)
    if norm == 0 or not np.isfinite(norm):
        return x
    return x / norm

## 8. Загрузка уже размеченных данных и выбор редких классов

In [9]:
def load_existing_labeled_datasets(paths):
    frames = []
    used_paths = []

    for path in paths:
        if path.exists():
            df = pd.read_csv(path)
            text_col = find_text_col(df)
            labels_col = find_labels_col(df)

            normalized = pd.DataFrame({
                TEXT_COL: df[text_col].astype(str),
                "labels": df[labels_col].astype(str),
            })
            if "evidence" in df.columns:
                normalized["evidence"] = df["evidence"].astype(str)
            else:
                normalized["evidence"] = ""
            normalized["source_file"] = str(path)
            normalized["text_hash"] = normalized[TEXT_COL].apply(text_hash)
            normalized["labels_list"] = normalized["labels"].apply(parse_labels_str)

            frames.append(normalized)
            used_paths.append(path)
            print("Загружен файл:", path, "shape=", df.shape)
        else:
            print("Файл не найден, пропускаю:", path)

    if not frames:
        raise FileNotFoundError(
            "Не найден ни один файл с уже размеченными данными. "
            "Проверь EXISTING_LABELED_PATHS."
        )

    all_df = pd.concat(frames, ignore_index=True)
    all_df = all_df.drop_duplicates(subset=["text_hash"]).reset_index(drop=True)
    return all_df, used_paths


existing_labeled_df, used_labeled_paths = load_existing_labeled_datasets(EXISTING_LABELED_PATHS)

class_counts_existing = {label: 0 for label in ALLOWED_LABELS}
for labels in existing_labeled_df["labels_list"]:
    for label in labels:
        if label in class_counts_existing:
            class_counts_existing[label] += 1

status_existing_df = pd.DataFrame([
    {
        "label": label,
        "count_existing": class_counts_existing[label],
        "need_to_target_min": max(0, TARGET_MIN_COUNT - class_counts_existing[label]),
    }
    for label in ALLOWED_LABELS
]).sort_values("count_existing")

display(status_existing_df)

if MANUAL_TARGET_LABELS is None:
    TARGET_LABELS = [
        label for label, count in class_counts_existing.items()
        if count < TARGET_MIN_COUNT
    ]
else:
    TARGET_LABELS = MANUAL_TARGET_LABELS.copy()

print("Целевые классы для добора:")
for label in TARGET_LABELS:
    print(f"- {label}: сейчас {class_counts_existing.get(label, 0)}, нужно добрать {max(0, TARGET_MIN_COUNT - class_counts_existing.get(label, 0))}")

Загружен файл: /content/drive/MyDrive/MLops_project/project/data/labeled/wb_feedbacks_llm_labeled_from_sample/balanced_50_per_class_random_chunks.csv shape= (146, 12)
Загружен файл: /content/drive/MyDrive/MLops_project/project/data/labeled/wb_feedbacks_llm_labeled_from_sample/balanced_50_per_class_random_chunks_final.csv shape= (146, 3)
Загружен файл: /content/drive/MyDrive/MLops_project/project/data/labeled/wb_feedbacks_llm_labeled_from_clusters/extra_from_hdbscan_candidate_clusters.csv shape= (866, 20)
Загружен файл: /content/drive/MyDrive/MLops_project/project/data/labeled/wb_feedbacks_llm_labeled_from_clusters/extra_from_hdbscan_candidate_clusters_final.csv shape= (866, 3)


,label,count_existing,need_to_target_min
2,Проблема с размером / посадкой,0,100
11,Нейтральный / информационный отзыв,0,100
10,Положительный отзыв,0,100
8,Цена / ценность,0,100
12,Другая проблема,0,100
9,Подделка / оригинальность,1,99
3,Несоответствие описанию,6,94
0,Брак / дефект товара,6,94
7,Проблема доставки / получения,7,93
1,Низкое качество материала,10,90


Целевые классы для добора:
- Подделка / оригинальность: сейчас 1, нужно добрать 99
- Нейтральный / информационный отзыв: сейчас 0, нужно добрать 100
- Проблема с размером / посадкой: сейчас 0, нужно добрать 100
- Низкое качество материала: сейчас 10, нужно добрать 90
- Цена / ценность: сейчас 0, нужно добрать 100
- Другая проблема: сейчас 0, нужно добрать 100


## 9. Загрузка HDBSCAN-кластеров и выбор кластеров-кандидатов

При настройке по умолчанию берутся именно те кластеры, которые в `cluster_llm_analysis.csv` были связаны с целевыми редкими классами.

In [10]:
if not CLUSTERS_PATH.exists():
    raise FileNotFoundError(
        f"Не найден файл с кластерами: {CLUSTERS_PATH}\n"
        "Сначала запусти ноутбук с HDBSCAN, который создает hdbscan_clusters.parquet."
    )

clusters_df = pd.read_parquet(CLUSTERS_PATH)
print("clusters_df shape:", clusters_df.shape)
print("clusters_df columns:", clusters_df.columns.tolist())

if TEXT_COL not in clusters_df.columns:
    raise ValueError(f"В clusters_df нет колонки {TEXT_COL}. Колонки: {clusters_df.columns.tolist()}")

if CLUSTER_COL not in clusters_df.columns:
    raise ValueError(f"В clusters_df нет колонки {CLUSTER_COL}. Колонки: {clusters_df.columns.tolist()}")

if PROB_COL not in clusters_df.columns:
    clusters_df[PROB_COL] = 1.0

if CLUSTER_ANALYSIS_CSV_PATH.exists():
    cluster_analysis_df = pd.read_csv(CLUSTER_ANALYSIS_CSV_PATH)
    print("cluster_analysis_df shape:", cluster_analysis_df.shape)
else:
    cluster_analysis_df = None
    print("cluster_llm_analysis.csv не найден. Будет доступен только режим USE_ONLY_CANDIDATE_CLUSTERS_FROM_ANALYSIS=False.")


def parse_matched_labels(value):
    return parse_labels_str(value)


def choose_candidate_clusters(cluster_analysis_df):
    if cluster_analysis_df is None:
        raise FileNotFoundError("cluster_analysis_df не загружен, нельзя выбрать candidate clusters по LLM-анализу.")

    df = cluster_analysis_df.copy()

    if "matched_labels" in df.columns:
        df["matched_labels_list"] = df["matched_labels"].apply(parse_matched_labels)
    elif "matched_labels_str" in df.columns:
        df["matched_labels_list"] = df["matched_labels_str"].apply(parse_matched_labels)
    else:
        df["matched_labels_list"] = [[] for _ in range(len(df))]

    if "matched_labels_str" not in df.columns:
        df["matched_labels_str"] = df["matched_labels_list"].apply(lambda xs: "; ".join(xs))

    if "confidence" in df.columns:
        df["confidence"] = pd.to_numeric(df["confidence"], errors="coerce").fillna(0.0)
    else:
        df["confidence"] = 0.0

    target_for_cluster_selection = TARGET_LABELS if SELECT_CLUSTERS_BY_TARGET_RARE_LABELS else ALLOWED_LABELS
    target_set = set(target_for_cluster_selection)

    df["target_matched_labels"] = df["matched_labels_list"].apply(
        lambda xs: [x for x in xs if x in target_set]
    )
    df["has_target_label"] = df["target_matched_labels"].apply(lambda xs: len(xs) > 0)

    base_mask = (
        df["has_target_label"]
        & (df["confidence"] >= MIN_CLUSTER_ANALYSIS_CONFIDENCE)
        & (df[CLUSTER_COL] != -1)
    )

    if INCLUDE_NEGATIVE_OR_MIXED_WITHOUT_MATCHED_LABELS and "positive_or_negative" in df.columns:
        pon = df["positive_or_negative"].astype(str).str.lower()
        fallback_mask = (
            pon.isin(["negative", "mixed"])
            & (df["confidence"] >= MIN_CLUSTER_ANALYSIS_CONFIDENCE)
            & (df[CLUSTER_COL] != -1)
        )
        mask = base_mask | fallback_mask
    else:
        mask = base_mask

    selected = df[mask].copy()

    if "cluster_size" in selected.columns:
        selected["cluster_size"] = pd.to_numeric(selected["cluster_size"], errors="coerce").fillna(0).astype(int)
        selected = selected.sort_values(["cluster_size", "confidence"], ascending=[False, False])
    else:
        sizes = clusters_df.groupby(CLUSTER_COL).size().rename("cluster_size").reset_index()
        selected = selected.merge(sizes, on=CLUSTER_COL, how="left")
        selected = selected.sort_values(["cluster_size", "confidence"], ascending=[False, False])

    if MAX_CLUSTERS_TO_PROCESS is not None:
        selected = selected.head(MAX_CLUSTERS_TO_PROCESS).copy()

    return selected


if USE_ONLY_CANDIDATE_CLUSTERS_FROM_ANALYSIS:
    candidate_clusters_df = choose_candidate_clusters(cluster_analysis_df)
    CANDIDATE_CLUSTER_IDS = candidate_clusters_df[CLUSTER_COL].astype(int).tolist()
    print("Выбрано кластеров-кандидатов:", len(CANDIDATE_CLUSTER_IDS))
else:
    candidate_clusters_df = pd.DataFrame()
    CANDIDATE_CLUSTER_IDS = sorted(clusters_df.loc[clusters_df[CLUSTER_COL] != -1, CLUSTER_COL].dropna().astype(int).unique().tolist())
    print("Используются все не-noise HDBSCAN-кластеры:", len(CANDIDATE_CLUSTER_IDS))

print("Всего отзывов в выбранных кластерах:", clusters_df[clusters_df[CLUSTER_COL].isin(CANDIDATE_CLUSTER_IDS)].shape[0])

if not candidate_clusters_df.empty:
    display_cols = [
        CLUSTER_COL,
        "cluster_size",
        "cluster_title",
        "cluster_feature",
        "main_theme",
        "matched_labels_str",
        "target_matched_labels",
        "positive_or_negative",
        "confidence",
        "evidence",
    ]
    display_cols = [col for col in display_cols if col in candidate_clusters_df.columns]
    display(candidate_clusters_df[display_cols].head(50))

clusters_df shape: (178132, 12)
clusters_df columns: ['nmId', 'productValuation', 'color', 'text', 'answer', 'source_row_idx', 'source_file', 'source_text_col', 'sample_part_id', 'sample_file', 'cluster_id', 'cluster_probability']
cluster_analysis_df shape: (278, 17)
Выбрано кластеров-кандидатов: 15
Всего отзывов в выбранных кластерах: 3293


,cluster_id,cluster_size,cluster_title,cluster_feature,main_theme,matched_labels_str,target_matched_labels,positive_or_negative,confidence,evidence
27,147,670,Продукт быстро вышел из строя,"Отзывы о товарах, которые сломались или начали...",Продукт быстро вышел из строя,Брак / дефект товара; Низкое качество материал...,[Низкое качество материала],negative,0.98,"Большинство отзывов содержат фразы о поломке, ..."
72,280,319,Препараты и БАДы для здоровья и самочувствия,"Отзывы касаются различных БАДов, витаминов и п...",Использование и оценка биологически активных д...,Положительный отзыв; Нейтральный / информацион...,[Нейтральный / информационный отзыв],mixed,0.85,"Отзывы содержат как положительные оценки (""рек..."
79,133,286,Костюмы с проблемами размерной сетки и качеством,"Отзывы касаются костюмов, где пользователи отм...",Проблемы с размером / посадкой и качество мате...,Проблема с размером / посадкой; Низкое качеств...,[Низкое качество материала],mixed,0.85,"Многие отзывы отмечают, что костюмы маломерят ..."
84,231,255,Неудовлетворенность товаром,"Отзывы выражают недовольство качеством, посадк...",Негативная обратная связь по характеристикам т...,Несоответствие ожиданиям / эффекту; Низкое кач...,[Низкое качество материала],negative,0.95,Большинство отзывов содержат фразы вроде 'не п...
97,295,235,Проблемы с товаром и возвраты,"Отзывы о товарах, в которых указаны различные ...",Несоответствие ожиданиям / эффекту,Брак / дефект товара; Несоответствие описанию;...,[Проблема с размером / посадкой],negative,0.95,Множество отзывов содержат упоминания о дефект...
96,15,235,Отзывы о свитерах,"Отзывы о свитерах, где пользователи описывают ...","Общие впечатления о свитерах, включая положите...",Несоответствие ожиданиям / эффекту; Проблема с...,"[Проблема с размером / посадкой, Низкое качест...",mixed,0.85,Отзывы содержат как положительные оценки (напр...
114,99,197,Положительные отзывы о толстовках,"Отзывы в основном положительные, акцентируются...",Покупатели довольны качеством и внешним видом ...,Положительный отзыв; Проблема с размером / пос...,[Проблема с размером / посадкой],positive,0.85,Большинство отзывов содержат слова вроде 'клас...
123,105,187,Отзывы о косметике для ресниц,Отзывы касаются качества и характеристик туши ...,Косметика для ресниц: положительные и негативн...,Несоответствие ожиданиям / эффекту; Низкое кач...,[Низкое качество материала],mixed,0.85,Отзывы содержат как положительные впечатления ...
131,82,170,"Отзывы о леггинсах: качество, посадка и удобство","Отзывы касаются качества, удобства, посадки и ...",Общая тема — оценка качества и удобства леггин...,Положительный отзыв; Несоответствие описанию; ...,[Низкое качество материала],mixed,0.85,"Большинство отзывов положительные, но есть упо..."
135,13,163,Положительные отзывы о туниках,"Отзывы в основном положительные, пользователи ...",Покупатели довольны качеством и внешним видом ...,Положительный отзыв; Проблема с размером / пос...,[Проблема с размером / посадкой],positive,0.85,Большинство отзывов выражают удовлетворенность...


## 10. Формирование датафрейма отзывов-кандидатов из выбранных HDBSCAN-кластеров

In [11]:
def build_candidate_reviews_df():
    candidate_df = clusters_df[clusters_df[CLUSTER_COL].isin(CANDIDATE_CLUSTER_IDS)].copy()

    if not candidate_clusters_df.empty:
        meta_cols = [
            CLUSTER_COL,
            "cluster_title",
            "cluster_feature",
            "main_theme",
            "matched_labels_str",
            "positive_or_negative",
            "confidence",
        ]
        meta_cols = [c for c in meta_cols if c in candidate_clusters_df.columns]
        if meta_cols:
            candidate_df = candidate_df.merge(
                candidate_clusters_df[meta_cols].drop_duplicates(subset=[CLUSTER_COL]),
                on=CLUSTER_COL,
                how="left",
                suffixes=("", "_cluster_analysis"),
            )

    rows_by_cluster = []
    for cluster_id in CANDIDATE_CLUSTER_IDS:
        part = candidate_df[candidate_df[CLUSTER_COL] == cluster_id].copy()
        if SORT_REVIEWS_BY_CLUSTER_PROBABILITY and PROB_COL in part.columns:
            part = part.sort_values(PROB_COL, ascending=False)
        if MAX_REVIEWS_PER_CLUSTER is not None:
            part = part.head(MAX_REVIEWS_PER_CLUSTER)
        rows_by_cluster.append(part)

    if rows_by_cluster:
        candidate_df = pd.concat(rows_by_cluster, ignore_index=True)
    else:
        candidate_df = pd.DataFrame(columns=candidate_df.columns)

    candidate_df[TEXT_COL] = candidate_df[TEXT_COL].astype(str)
    candidate_df["text_hash"] = candidate_df[TEXT_COL].apply(text_hash)
    candidate_df = candidate_df.drop_duplicates(subset=["text_hash"]).reset_index(drop=True)

    # Убираем уже размеченные тексты.
    if DEDUP_BY_TEXT:
        existing_hashes = set(existing_labeled_df["text_hash"].tolist())
        before = len(candidate_df)
        candidate_df = candidate_df[~candidate_df["text_hash"].isin(existing_hashes)].copy()
        print("Удалено дубликатов с уже размеченными данными:", before - len(candidate_df))

    candidate_df = candidate_df.reset_index(drop=True)
    candidate_df["candidate_idx"] = np.arange(len(candidate_df))
    return candidate_df


candidate_reviews_df = build_candidate_reviews_df()
print("Отзывов-кандидатов после фильтрации и дедупликации:", len(candidate_reviews_df))
display(candidate_reviews_df[[TEXT_COL, CLUSTER_COL, PROB_COL]].head(10))

Удалено дубликатов с уже размеченными данными: 0
Отзывов-кандидатов после фильтрации и дедупликации: 3290


,text,cluster_id,cluster_probability
0,Сломались после недели носки.,147,1.0
1,"Был запах краски, очень силтный, но выветрился...",147,1.0
2,Спустя полтора сезона лопнула подошва на двух ...,147,1.0
3,"Бесшумный, свою функцию выполняет, сломался че...",147,1.0
4,Резиновая ручка внизу отвалилась через 3недели,147,1.0
5,За 2 месяца использования наполнитель сбился в...,147,1.0
6,Спустя 10 дней перестало работать колесико,147,1.0
7,"ужасная подводка,высохла через 2 дня, зря выбр...",147,1.0
8,"Честно сказать после первой стирки,как тряпка",147,1.0
9,через пару недель замок сломался,147,1.0


## 11. Описания классов для эмбеддингов

Эти описания нужны, чтобы эмбеддинг класса учитывал не только уже найденные примеры, но и смысл класса из prompt.

In [12]:
CLASS_DESCRIPTIONS = {
    "Брак / дефект товара": "Класс: Брак / дефект товара. Отзыв о том, что товар сломан, поврежден, не работает, работает неправильно, быстро сломался, имеет трещины, сколы, заводской дефект или явную неисправность. Типичные слова: сломан, не работает, брак, дефект, трещина, поврежден, перестал работать.",
    "Низкое качество материала": "Класс: Низкое качество материала. Покупатель жалуется на плохой материал, дешевый пластик, плохую ткань, кривые швы, хлипкость, слабую сборку, неприятный материал, быструю изнашиваемость. Типичные слова: плохой материал, хлипкий, дешевый пластик, ткань плохая, швы кривые, развалилось, качество ужасное.",
    "Проблема с размером / посадкой": "Класс: Проблема с размером / посадкой. Товар не подошел по размеру, форме, посадке или совместимости: маломерит, большемерит, неудобно сидит, не подошел к устройству, оказался меньше или больше ожидаемого. Типичные слова: маломерит, большемерит, не подошел, мал, велик, размер не тот, не налез, сидит плохо.",
    "Несоответствие описанию": "Класс: Несоответствие описанию. Товар отличается от карточки товара, фото, описания, характеристик или заявленных свойств: другой цвет, другой материал, другая модель, не те характеристики, описание вводит в заблуждение.",
    "Несоответствие ожиданиям / эффекту": "Класс: Несоответствие ожиданиям / эффекту. Товар формально пришел и не сломан, но не выполняет ожидаемую функцию или не дает результата: средство не помогает, устройство слабое, товар бесполезен, эффект отсутствует, результат хуже ожидаемого.",
    "Проблема с комплектацией": "Класс: Проблема с комплектацией. В заказе не хватает деталей, аксессуаров, инструкции, кабеля, насадок, креплений или других элементов комплекта; пришла только часть набора; комплект неполный.",
    "Проблема с упаковкой": "Класс: Проблема с упаковкой. Жалоба относится к упаковке: коробка мятая, упаковка вскрыта, порвана, плохая защита, товар был плохо упакован.",
    "Проблема доставки / получения": "Класс: Проблема доставки / получения. Проблема связана с доставкой или получением: долго ехал, задержали, перенесли доставку, пришел не туда, проблемы с пунктом выдачи, курьером, логистикой, заказ потеряли.",
    "Цена / ценность": "Класс: Цена / ценность. Покупатель пишет, что товар не стоит своих денег, цена завышена, качество не соответствует цене, можно найти дешевле, за эти деньги ожидал большего. Типичные слова: дорого, не стоит денег, цена завышена, своих денег не стоит, деньги на ветер, за такую цену плохо.",
    "Подделка / оригинальность": "Класс: Подделка / оригинальность. Покупатель сомневается в подлинности товара или пишет, что товар похож на подделку, неоригинальный, не фирменный, отличается от оригинала, вызывает сомнения бренд, маркировка или упаковка. Типичные слова: подделка, не оригинал, паль, фейк, копия, не фирменный, отличается от оригинала.",
    "Положительный отзыв": "Класс: Положительный отзыв. Покупатель явно доволен товаром: товар понравился, все хорошо, рекомендую, качество устраивает, покупкой доволен, товар оправдал ожидания.",
    "Нейтральный / информационный отзыв": "Класс: Нейтральный / информационный отзыв. Отзыв не содержит явной положительной или отрицательной оценки, а только сообщает факт: получил, пока не использовал, дополню позже, купил в подарок, не открывал, пока ничего сказать не могу. Типичные слова: получил, пока не пользовался, позже дополню, купил в подарок, еще не открывал.",
    "Другая проблема": "Класс: Другая проблема. В отзыве есть негативная проблема, но ее нельзя уверенно отнести ни к одному из основных классов. Непонятная негативная жалоба, запасной класс для проблем без четкой категории.",
}

# Дополнительные поисковые формулировки для классов, которые собирались медленно.
# Это не финальная разметка. Эти тексты нужны только для семантического поиска кандидатов.
# Финальное решение по-прежнему принимает LLM через основной USER_PROMPT.
CLASS_SEARCH_PROTOTYPES = {
    "Подделка / оригинальность": [
        "отзыв: это подделка, товар не оригинальный",
        "покупатель пишет, что пришел не оригинал",
        "похоже на фейк или копию бренда",
        "сомнения в оригинальности товара",
        "товар отличается от оригинального, нет фирменных признаков",
        "подозрение на паль, подделку, копию, не фирменный товар",
        "упаковка, маркировка или бренд вызывают сомнения в подлинности",
        "продавец прислал не оригинальный товар",
    ],
    "Нейтральный / информационный отзыв": [
        "отзыв: товар получил, пока не пользовался",
        "пока ничего сказать не могу, позже дополню отзыв",
        "еще не открывал и не проверял",
        "купил в подарок, сам не использовал",
        "доставка получена, опыт использования пока отсутствует",
        "на вид нормально, в работе пока не проверял",
        "первое впечатление без оценки качества",
        "просто сообщил факт получения товара",
    ],
    "Цена / ценность": [
        "отзыв: товар не стоит своих денег",
        "цена завышена для такого качества",
        "дорого, можно найти дешевле",
        "качество не соответствует цене",
        "деньги на ветер, покупка не оправдала стоимость",
        "за такую цену ожидал большего",
        "плохое соотношение цены и качества",
        "слишком дорого для такого товара",
    ],
    "Другая проблема": [
        "отзыв: есть проблема, но она не про упаковку, доставку, размер, комплектацию или описание",
        "неприятный запах, химический запах, странный запах товара",
        "товар вызывает раздражение, дискомфорт, аллергическую реакцию",
        "неудобно пользоваться, неудобная конструкция, странное использование",
        "плохая инструкция, непонятно как пользоваться",
        "товар шумит, скрипит, пахнет, пачкает, оставляет следы",
        "негативная жалоба без очевидной категории",
        "есть минус или проблема, но неясно к какому классу ее отнести",
    ],
    "Проблема с размером / посадкой": [
        "отзыв: товар маломерит или большемерит",
        "размер не подошел, оказался маленьким или большим",
        "не подошел по форме, посадке или совместимости",
        "не налезает, сидит плохо, неудобная посадка",
        "размерная сетка не совпадает",
        "чехол, деталь или аксессуар не подошел к устройству",
    ],
    "Низкое качество материала": [
        "отзыв: плохой материал, дешевый материал",
        "хлипкий товар, слабая сборка",
        "кривые швы, плохая ткань, торчат нитки",
        "дешевый пластик, неприятный материал",
        "материал быстро износился или развалился",
        "качество изготовления ужасное",
    ],
}

for label in TARGET_LABELS:
    print("=", label)
    print(CLASS_DESCRIPTIONS[label][:300], "...")
    if label in CLASS_SEARCH_PROTOTYPES:
        print("Дополнительных поисковых прототипов:", len(CLASS_SEARCH_PROTOTYPES[label]))


= Подделка / оригинальность
Класс: Подделка / оригинальность. Покупатель сомневается в подлинности товара или пишет, что товар похож на подделку, неоригинальный, не фирменный, отличается от оригинала, вызывает сомнения бренд, маркировка или упаковка. Типичные слова: подделка, не оригинал, паль, фейк, копия, не фирменный, отлич ...
Дополнительных поисковых прототипов: 8
= Нейтральный / информационный отзыв
Класс: Нейтральный / информационный отзыв. Отзыв не содержит явной положительной или отрицательной оценки, а только сообщает факт: получил, пока не использовал, дополню позже, купил в подарок, не открывал, пока ничего сказать не могу. Типичные слова: получил, пока не пользовался, позже дополню, купи ...
Дополнительных поисковых прототипов: 8
= Проблема с размером / посадкой
Класс: Проблема с размером / посадкой. Товар не подошел по размеру, форме, посадке или совместимости: маломерит, большемерит, неудобно сидит, не подошел к устройству, оказался меньше или больше ожидаемого. Типичные

## 12. Загрузка модели эмбеддингов

In [13]:
embedding_model = SentenceTransformer(EMBEDDING_MODEL_NAME)
print("Embedding model loaded:", EMBEDDING_MODEL_NAME)

modules.json:   0%|          | 0.00/387 [00:00<?, ?B/s]

README.md:   0%|          | 0.00/179k [00:00<?, ?B/s]

sentence_bert_config.json:   0%|          | 0.00/57.0 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/694 [00:00<?, ?B/s]

model.safetensors:   0%|          | 0.00/1.11G [00:00<?, ?B/s]

...

sentencepiece.bpe.model:   0%|          | 0.00/5.07M [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/17.1M [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/280 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/200 [00:00<?, ?B/s]

Embedding model loaded: intfloat/multilingual-e5-base


## 13. Функции для расчета эмбеддингов

In [14]:
def prepare_for_embedding(texts, kind="passage"):
    texts = [str(x).strip() for x in texts]
    if not USE_E5_PREFIXES:
        return texts

    if "e5" not in EMBEDDING_MODEL_NAME.lower():
        return texts

    prefix = "query: " if kind == "query" else "passage: "
    return [prefix + x for x in texts]


def encode_texts(texts, kind="passage", batch_size=EMBEDDING_BATCH_SIZE, desc="Encoding"):
    prepared = prepare_for_embedding(texts, kind=kind)
    embeddings = embedding_model.encode(
        prepared,
        batch_size=batch_size,
        show_progress_bar=True,
        normalize_embeddings=True,
        convert_to_numpy=True,
    )
    return embeddings.astype("float32")


def get_examples_for_label(label, max_examples=None):
    mask = existing_labeled_df["labels_list"].apply(lambda xs: label in xs)
    texts = existing_labeled_df.loc[mask, TEXT_COL].dropna().astype(str).tolist()
    if max_examples is not None:
        texts = texts[:max_examples]
    return texts

## 14. Построение эмбеддингов целевых классов

Для каждого класса строится комбинированный вектор:

```text
0.7 × центроид уже размеченных примеров + 0.3 × описание класса
```

Если примеров класса слишком мало, используется только описание класса.

In [15]:
def make_prototype_text(label, text, source):
    """Делаем query-текст для поиска похожих отзывов."""
    text = str(text).strip()
    if source == "existing_example":
        return f"Найди похожий отзыв для класса: {label}. Пример отзыва: {text}"
    return text


def build_class_prototype_embeddings():
    """
    Строит несколько эмбеддингов-прототипов для каждого класса.

    Раньше для класса строился один эмбеддинг: центроид примеров + описание.
    Для редких и сложных классов это часто плохо: один усредненный вектор смешивает разные формулировки.
    Теперь для каждого класса используется несколько query-прототипов, а близость кандидата к классу
    считается как максимум близости до этих прототипов.
    """
    rng = np.random.default_rng(RANDOM_SEED_FOR_EXAMPLE_PROTOTYPES)
    prototype_rows = []
    prototype_texts = []

    for label in TARGET_LABELS:
        if USE_CLASS_DESCRIPTION_AS_PROTOTYPE:
            prototype_rows.append({
                "label": label,
                "prototype_source": "class_description",
                "prototype_text": CLASS_DESCRIPTIONS[label],
            })
            prototype_texts.append(CLASS_DESCRIPTIONS[label])

        # Специальные поисковые формулировки особенно важны для медленных классов.
        for proto in CLASS_SEARCH_PROTOTYPES.get(label, []):
            prototype_rows.append({
                "label": label,
                "prototype_source": "manual_search_prototype",
                "prototype_text": proto,
            })
            prototype_texts.append(proto)

        if USE_EXISTING_EXAMPLES_AS_PROTOTYPES:
            example_texts = get_examples_for_label(label)
            if len(example_texts) > MAX_EXAMPLES_AS_PROTOTYPES_PER_LABEL:
                idx = rng.choice(len(example_texts), size=MAX_EXAMPLES_AS_PROTOTYPES_PER_LABEL, replace=False)
                example_texts = [example_texts[i] for i in idx]

            for ex_text in example_texts:
                proto_text = make_prototype_text(label, ex_text, source="existing_example")
                prototype_rows.append({
                    "label": label,
                    "prototype_source": "existing_example",
                    "prototype_text": proto_text,
                })
                prototype_texts.append(proto_text)

    prototype_meta_df = pd.DataFrame(prototype_rows)
    if prototype_meta_df.empty:
        raise ValueError("Не получилось построить ни одного prototype embedding")

    prototype_embeddings = encode_texts(
        prototype_texts,
        kind="query",
        desc="class search prototypes",
    ).astype("float32")

    np.save(CLASS_PROTOTYPE_EMBEDDINGS_NPY_PATH, prototype_embeddings)
    prototype_meta_df.to_csv(CLASS_PROTOTYPE_META_CSV_PATH, index=False, encoding="utf-8-sig")

    debug_df = (
        prototype_meta_df
        .groupby(["label", "prototype_source"])
        .size()
        .reset_index(name="n_prototypes")
        .sort_values(["label", "prototype_source"])
    )
    return prototype_embeddings, prototype_meta_df, debug_df


class_prototype_embeddings, class_prototype_meta_df, class_debug_df = build_class_prototype_embeddings()
display(class_debug_df)
print("class_prototype_embeddings shape:", class_prototype_embeddings.shape)
print("CLASS_PROTOTYPE_META_CSV_PATH:", CLASS_PROTOTYPE_META_CSV_PATH)


Batches:   0%|          | 0/1 [00:00<?, ?it/s]

,label,prototype_source,n_prototypes
0,Другая проблема,class_description,1
1,Другая проблема,manual_search_prototype,8
2,Нейтральный / информационный отзыв,class_description,1
3,Нейтральный / информационный отзыв,manual_search_prototype,8
4,Низкое качество материала,class_description,1
5,Низкое качество материала,existing_example,10
6,Низкое качество материала,manual_search_prototype,6
7,Подделка / оригинальность,class_description,1
8,Подделка / оригинальность,existing_example,1
9,Подделка / оригинальность,manual_search_prototype,8


class_prototype_embeddings shape: (61, 768)
CLASS_PROTOTYPE_META_CSV_PATH: /content/drive/MyDrive/MLops_project/project/data/labeled/wb_feedbacks_llm_labeled_from_embeddings/embedding_cache/class_prototype_meta.csv


## 15. Эмбеддинги отзывов-кандидатов

Если кэш уже есть, он будет загружен. Если поменялись кандидаты или настройки, удали файлы из `embedding_cache` или поменяй `OUTPUT_BASENAME`.

In [16]:
def load_or_compute_candidate_embeddings(candidate_reviews_df):
    can_use_cache = (
        CANDIDATE_EMBEDDINGS_NPY_PATH.exists()
        and CANDIDATE_META_PARQUET_PATH.exists()
    )

    if can_use_cache:
        cached_meta = pd.read_parquet(CANDIDATE_META_PARQUET_PATH)
        same_len = len(cached_meta) == len(candidate_reviews_df)
        same_hashes = same_len and cached_meta["text_hash"].tolist() == candidate_reviews_df["text_hash"].tolist()

        if same_hashes:
            print("Загружаю эмбеддинги кандидатов из кэша:", CANDIDATE_EMBEDDINGS_NPY_PATH)
            embeddings = np.load(CANDIDATE_EMBEDDINGS_NPY_PATH)
            return embeddings.astype("float32")
        else:
            print("Кэш найден, но кандидаты изменились. Пересчитываю эмбеддинги.")

    texts = candidate_reviews_df[TEXT_COL].astype(str).tolist()
    embeddings = encode_texts(texts, kind="passage", desc="candidate reviews")
    np.save(CANDIDATE_EMBEDDINGS_NPY_PATH, embeddings)
    candidate_reviews_df.to_parquet(CANDIDATE_META_PARQUET_PATH, index=False)
    print("Эмбеддинги сохранены:", CANDIDATE_EMBEDDINGS_NPY_PATH)
    return embeddings.astype("float32")


candidate_embeddings = load_or_compute_candidate_embeddings(candidate_reviews_df)
print("candidate_embeddings shape:", candidate_embeddings.shape)

Загружаю эмбеддинги кандидатов из кэша: /content/drive/MyDrive/MLops_project/project/data/labeled/wb_feedbacks_llm_labeled_from_embeddings/embedding_cache/candidate_embeddings.npy
candidate_embeddings shape: (3290, 768)


## 16. Поиск ближайших отзывов к редким классам

Здесь формируется общий список кандидатов для LLM-разметки. Один и тот же отзыв может быть близок сразу к нескольким редким классам, поэтому дубликаты объединяются.

In [17]:
def build_semantic_candidate_pool():
    rows = []

    for label in TARGET_LABELS:
        proto_indices = class_prototype_meta_df.index[class_prototype_meta_df["label"] == label].to_numpy()
        if len(proto_indices) == 0:
            continue

        label_proto_embs = class_prototype_embeddings[proto_indices]

        # cosine similarity, так как embeddings уже normalize_embeddings=True
        # Для класса берем максимум по нескольким прототипам.
        sims_matrix = candidate_embeddings @ label_proto_embs.T
        sims = sims_matrix.max(axis=1)
        best_proto_local_idx = sims_matrix.argmax(axis=1)
        best_proto_global_idx = proto_indices[best_proto_local_idx]

        top_k_for_label = min(TOP_K_BY_LABEL.get(label, TOP_K_PER_CLASS), len(sims))
        if top_k_for_label == 0:
            continue

        top_indices = np.argpartition(-sims, kth=top_k_for_label - 1)[:top_k_for_label]
        top_indices = top_indices[np.argsort(-sims[top_indices])]

        for rank, idx in enumerate(top_indices, start=1):
            best_proto_idx = int(best_proto_global_idx[idx])
            proto_row = class_prototype_meta_df.loc[best_proto_idx]
            rows.append({
                "candidate_idx": int(idx),
                "target_label_for_search": label,
                "semantic_rank": rank,
                "semantic_similarity": float(sims[idx]),
                "best_prototype_source": proto_row.get("prototype_source", ""),
                "best_prototype_text": proto_row.get("prototype_text", ""),
                "target_need_before_run": max(0, TARGET_MIN_COUNT - class_counts_existing.get(label, 0)),
            })

    pool_df = pd.DataFrame(rows)
    if pool_df.empty:
        return pool_df

    # Для одного текста оставляем лучшую близость, но сохраняем список классов, по которым он был найден.
    # Важно: сортируем не только по similarity, но и по дефициту класса. Так самые редкие классы
    # проверяются раньше, а не тонут под более похожими, но уже почти закрытыми классами.
    agg_df = (
        pool_df
        .groupby("candidate_idx")
        .agg(
            best_semantic_similarity=("semantic_similarity", "max"),
            best_semantic_rank=("semantic_rank", "min"),
            max_target_need_before_run=("target_need_before_run", "max"),
            search_target_labels=("target_label_for_search", lambda xs: "; ".join(sorted(set(xs)))),
            best_prototype_sources=("best_prototype_source", lambda xs: "; ".join(sorted(set(map(str, xs))))),
            best_prototype_texts=("best_prototype_text", lambda xs: " || ".join(list(dict.fromkeys(map(str, xs)))[:3])),
        )
        .reset_index()
    )

    result_df = agg_df.merge(candidate_reviews_df, on="candidate_idx", how="left")
    result_df = result_df.sort_values(
        ["max_target_need_before_run", "best_semantic_similarity", "best_semantic_rank"],
        ascending=[False, False, True],
    ).reset_index(drop=True)

    return result_df


semantic_candidate_pool_df = build_semantic_candidate_pool()
print("Уникальных semantic candidates:", len(semantic_candidate_pool_df))
display_cols = [
    "max_target_need_before_run",
    "best_semantic_similarity",
    "best_semantic_rank",
    "search_target_labels",
    "best_prototype_sources",
    TEXT_COL,
    CLUSTER_COL,
    PROB_COL,
    "cluster_title",
    "matched_labels_str",
]
display_cols = [c for c in display_cols if c in semantic_candidate_pool_df.columns]
display(semantic_candidate_pool_df[display_cols].head(30))


Уникальных semantic candidates: 2891


,max_target_need_before_run,best_semantic_similarity,best_semantic_rank,search_target_labels,best_prototype_sources,text,cluster_id,cluster_probability,cluster_title,matched_labels_str
0,100,0.880090,1,Другая проблема; Нейтральный / информационный ...,class_description; manual_search_prototype,"Качество материала низкое, ткань грубая. Отказ",295,0.859571,Проблемы с товаром и возвраты,Брак / дефект товара; Несоответствие описанию;...
1,100,0.878629,2,Другая проблема; Нейтральный / информационный ...,class_description; existing_example; manual_se...,Отказ. Качество шитья и материал не понравились,295,0.850911,Проблемы с товаром и возвраты,Брак / дефект товара; Несоответствие описанию;...
2,100,0.875821,3,Другая проблема; Нейтральный / информационный ...,class_description; existing_example; manual_se...,Очень дешёвая ткань на вид. Отказ,295,0.878014,Проблемы с товаром и возвраты,Брак / дефект товара; Несоответствие описанию;...
3,100,0.875528,1,Другая проблема; Нейтральный / информационный ...,class_description; existing_example; manual_se...,Сумка с ужасным запахом. Возврат,295,0.907649,Проблемы с товаром и возвраты,Брак / дефект товара; Несоответствие описанию;...
4,100,0.875237,2,Другая проблема; Нейтральный / информационный ...,class_description; existing_example; manual_se...,Пришёл не тот товар. Отказ,295,0.839801,Проблемы с товаром и возвраты,Брак / дефект товара; Несоответствие описанию;...
5,100,0.871899,2,Другая проблема; Нейтральный / информационный ...,class_description; manual_search_prototype,Неудобный. Отказ,295,0.806802,Проблемы с товаром и возвраты,Брак / дефект товара; Несоответствие описанию;...
6,100,0.871256,3,Другая проблема; Нейтральный / информационный ...,class_description; existing_example; manual_se...,"С якорем, ужасный запах, сшиты криво. Возврат",295,1.000000,Проблемы с товаром и возвраты,Брак / дефект товара; Несоответствие описанию;...
7,100,0.870943,1,Другая проблема; Нейтральный / информационный ...,class_description; manual_search_prototype,Сидит плохо. Фасон несуразной формы,295,0.783399,Проблемы с товаром и возвраты,Брак / дефект товара; Несоответствие описанию;...
8,100,0.870447,4,Другая проблема; Нейтральный / информационный ...,class_description; existing_example; manual_se...,Качество дешевое. Отказ,295,1.000000,Проблемы с товаром и возвраты,Брак / дефект товара; Несоответствие описанию;...
9,100,0.869411,2,Другая проблема; Нейтральный / информационный ...,class_description; existing_example; manual_se...,Пришли со в скрытой упаковкой. Отказ,295,0.839801,Проблемы с товаром и возвраты,Брак / дефект товара; Несоответствие описанию;...


## 17. LLM-разметка найденных semantic candidates

In [18]:
def safe_json_loads(content):
    content = str(content).strip()
    try:
        return json.loads(content)
    except json.JSONDecodeError:
        start = content.find("{")
        end = content.rfind("}")
        if start != -1 and end != -1 and end > start:
            return json.loads(content[start:end + 1])
        raise


def build_full_prompt(review_text):
    allowed_labels_text = "\n".join(f'- "{label}"' for label in ALLOWED_LABELS)
    review_text_json = json.dumps(str(review_text), ensure_ascii=False)

    return f"""
{USER_PROMPT}

Разрешенный список классов, который можно использовать в поле labels:
{allowed_labels_text}

Отзыв для классификации:
{review_text_json}

Верни ответ строго в JSON формате:

{{
  "labels": ["..."],
  "confidence": 0.0,
  "evidence": "короткое объяснение, почему выбраны эти классы"
}}

Жесткие требования к ответу:
- Верни только один JSON-объект.
- Не возвращай markdown.
- Не возвращай текст вне JSON.
- Не добавляй комментарии до или после JSON.
- Поле labels должно быть массивом строк.
- labels может содержать только классы из разрешенного списка.
- Если выбрано "Нейтральный / информационный отзыв", это должен быть единственный класс.
- Если выбрано "Другая проблема", не добавляй ее вместе с конкретным проблемным классом.
- confidence — число от 0 до 1.
- evidence — короткое объяснение на русском.
""".strip()


def normalize_model_response(content):
    data = safe_json_loads(content)

    labels = data.get("labels", [])
    if not isinstance(labels, list):
        labels = []

    normalized_labels = []
    for label in labels:
        label = str(label).strip()
        if label in ALLOWED_LABELS and label not in normalized_labels:
            normalized_labels.append(label)

    # Нейтральный класс не должен сочетаться с другими.
    if "Нейтральный / информационный отзыв" in normalized_labels and len(normalized_labels) > 1:
        normalized_labels = ["Нейтральный / информационный отзыв"]

    # "Другая проблема" убирается, если есть конкретный проблемный класс.
    if "Другая проблема" in normalized_labels:
        has_concrete_problem = any(label in PROBLEM_LABELS for label in normalized_labels)
        if has_concrete_problem:
            normalized_labels = [x for x in normalized_labels if x != "Другая проблема"]

    if len(normalized_labels) == 0:
        return {
            "labels": [],
            "confidence": 0.0,
            "evidence": "",
            "raw_response": str(content),
            "error": "no_valid_labels",
        }

    try:
        confidence = float(data.get("confidence", 0.0))
    except Exception:
        confidence = 0.0
    confidence = max(0.0, min(1.0, confidence))

    return {
        "labels": normalized_labels,
        "confidence": confidence,
        "evidence": str(data.get("evidence", "")),
        "raw_response": str(content),
        "error": None,
    }


def classify_review(
    review_text,
    model_name=MODEL_NAME,
    temperature=TEMPERATURE,
    max_retries=MAX_RETRIES,
):
    full_prompt = build_full_prompt(review_text)

    for attempt in range(max_retries):
        try:
            response = client.chat.completions.create(
                model=model_name,
                messages=[
                    {
                        "role": "system",
                        "content": (
                            "/no_think\n"
                            "Ты ассистент для строгой разметки данных для обучения ML-классификатора. "
                            "Твоя цель — высокая точность разметки, а не красивое объяснение. "
                            "Отвечай только валидным JSON-объектом. "
                            "Не добавляй markdown, рассуждения, комментарии или текст вне JSON."
                        ),
                    },
                    {
                        "role": "user",
                        "content": full_prompt,
                    },
                ],
                temperature=temperature,
                response_format={"type": "json_object"},
            )

            content = response.choices[0].message.content
            return normalize_model_response(content)

        except Exception as e:
            error_text = str(e)

            if (
                "429" in error_text
                or "rate limit" in error_text.lower()
                or "tokens per day" in error_text.lower()
                or "TPD" in error_text
            ):
                print("\n" + "=" * 80)
                print("ДОСТИГНУТ ЛИМИТ API / RATE LIMIT")
                print(f"attempt: {attempt + 1}/{max_retries}")
                print(error_text)
                print("=" * 80 + "\n")

            if attempt == max_retries - 1:
                return {
                    "labels": [],
                    "confidence": 0.0,
                    "evidence": "",
                    "raw_response": "",
                    "error": error_text,
                }

            sleep_seconds = RETRY_BASE_SLEEP_SECONDS * (2 ** attempt)
            time.sleep(sleep_seconds)

    return {
        "labels": [],
        "confidence": 0.0,
        "evidence": "",
        "raw_response": "",
        "error": "unknown_error",
    }

## 18. Сохранение, продолжение после остановки и прогресс по классам

In [19]:
def load_existing_text_hashes_and_output():
    existing_hashes = set(existing_labeled_df["text_hash"].tolist()) if DEDUP_BY_TEXT else set()
    already_saved_rows = []

    if RESUME_FROM_EXISTING_OUTPUT and OUTPUT_CSV_PATH.exists():
        current_df = pd.read_csv(OUTPUT_CSV_PATH)
        already_saved_rows = current_df.to_dict("records")
        if TEXT_COL in current_df.columns:
            for text in current_df[TEXT_COL].dropna().astype(str):
                existing_hashes.add(text_hash(text))
        print("Найден текущий результат для продолжения:", OUTPUT_CSV_PATH)
        print("Уже сохранено строк в текущем результате:", len(already_saved_rows))
    else:
        print("Текущий результат не найден. Новый добор начнется с нуля.")

    return existing_hashes, already_saved_rows


def append_example_to_disk(example_row):
    full_row_df = pd.DataFrame([example_row])

    full_row_df.to_csv(
        OUTPUT_CSV_PATH,
        mode="a",
        header=not OUTPUT_CSV_PATH.exists(),
        index=False,
        encoding="utf-8-sig",
    )

    final_row_df = pd.DataFrame([{
        "отзыв": example_row.get(TEXT_COL, ""),
        "labels": example_row.get("labels_str", ""),
        "evidence": example_row.get("evidence", ""),
    }])

    final_row_df.to_csv(
        FINAL_CSV_PATH,
        mode="a",
        header=not FINAL_CSV_PATH.exists(),
        index=False,
        encoding="utf-8-sig",
    )


def save_full_outputs(rows):
    df = pd.DataFrame(rows)
    if not df.empty:
        df.to_parquet(OUTPUT_PARQUET_PATH, index=False)
    return df


def build_total_class_status_df(rows):
    current_counts = class_counts_existing.copy()
    extra_counts = {label: 0 for label in ALLOWED_LABELS}

    for row in rows:
        labels_counted = row.get("labels_counted", [])
        if isinstance(labels_counted, str):
            labels_counted = parse_labels_str(labels_counted)
        for label in labels_counted:
            if label in current_counts:
                current_counts[label] += 1
                extra_counts[label] += 1

    status_df = pd.DataFrame([
        {
            "label": label,
            "count_existing_before_this_notebook": class_counts_existing[label],
            "count_added_by_this_notebook": extra_counts[label],
            "count_total_after_this_notebook": current_counts[label],
            "need_to_target_min": max(0, TARGET_MIN_COUNT - current_counts[label]),
            "is_target_label": label in TARGET_LABELS,
        }
        for label in ALLOWED_LABELS
    ]).sort_values(["is_target_label", "count_total_after_this_notebook"], ascending=[False, True])

    status_df.to_csv(STATUS_CSV_PATH, index=False, encoding="utf-8-sig")
    return status_df


def target_counts_reached(rows):
    status_df = build_total_class_status_df(rows)
    target_status = status_df[status_df["label"].isin(TARGET_LABELS)]
    return (target_status["count_total_after_this_notebook"] >= TARGET_MIN_COUNT).all()

## 19. Основной цикл добора

Запусти эту ячейку для начала LLM-разметки semantic candidates.

In [20]:
def collect_extra_examples_from_embedding_candidates():
    if RESET_OUTPUT_FILES_ON_START:
        for path in [OUTPUT_CSV_PATH, OUTPUT_PARQUET_PATH, FINAL_CSV_PATH, STATUS_CSV_PATH, RUN_STATS_JSON_PATH]:
            if path.exists():
                path.unlink()
        print("Старые выходные файлы удалены.")

    existing_hashes, selected_examples = load_existing_text_hashes_and_output()

    sent_to_model = 0
    scanned_reviews = 0
    skipped_duplicates = 0
    skipped_empty = 0
    skipped_without_target_label = 0
    saved_examples_before = len(selected_examples)

    current_total_counts = class_counts_existing.copy()
    for row in selected_examples:
        labels_counted = row.get("labels_counted", row.get("labels", []))
        if isinstance(labels_counted, str):
            labels_counted = parse_labels_str(labels_counted)
        for label in labels_counted:
            if label in current_total_counts:
                current_total_counts[label] += 1

    class_bars = {}
    for pos, label in enumerate(TARGET_LABELS):
        current_count = current_total_counts[label]
        class_bars[label] = tqdm(
            total=TARGET_MIN_COUNT,
            initial=min(current_count, TARGET_MIN_COUNT),
            desc=label[:35],
            position=pos,
            leave=True,
        )
        class_bars[label].set_postfix({"count": current_count, "need": max(0, TARGET_MIN_COUNT - current_count)})

    main_bar = tqdm(
        semantic_candidate_pool_df.iterrows(),
        total=len(semantic_candidate_pool_df),
        desc="Semantic candidates",
        position=len(TARGET_LABELS),
        leave=True,
    )

    try:
        for source_row_idx, row in main_bar:
            if MAX_REVIEWS_TO_SEND_TO_MODEL is not None and sent_to_model >= MAX_REVIEWS_TO_SEND_TO_MODEL:
                print("Достигнут MAX_REVIEWS_TO_SEND_TO_MODEL")
                break

            if STOP_WHEN_ALL_TARGETS_REACHED:
                if all(current_total_counts[label] >= TARGET_MIN_COUNT for label in TARGET_LABELS):
                    print("Все целевые классы достигли TARGET_MIN_COUNT")
                    break

            review_text = str(row.get(TEXT_COL, "")).strip()
            if not review_text or review_text.lower() == "nan":
                skipped_empty += 1
                continue

            current_hash = text_hash(review_text)
            if DEDUP_BY_TEXT and current_hash in existing_hashes:
                skipped_duplicates += 1
                continue

            scanned_reviews += 1
            sent_to_model += 1

            result = classify_review(review_text)
            labels = result.get("labels", [])
            labels_counted = [label for label in labels if label in ALLOWED_LABELS]
            target_labels_found = [label for label in labels_counted if label in TARGET_LABELS]

            if SAVE_ONLY_IF_HAS_TARGET_LABEL and not target_labels_found:
                skipped_without_target_label += 1
                existing_hashes.add(current_hash)
                if REQUEST_SLEEP_SECONDS > 0:
                    time.sleep(REQUEST_SLEEP_SECONDS)
                continue

            if labels_counted:
                example_row = {
                    TEXT_COL: review_text,
                    "labels": labels,
                    "labels_str": "; ".join(labels),
                    "labels_counted": labels_counted,
                    "labels_counted_str": "; ".join(labels_counted),
                    "target_labels_found": target_labels_found,
                    "target_labels_found_str": "; ".join(target_labels_found),
                    "confidence": result.get("confidence", 0.0),
                    "evidence": result.get("evidence", ""),
                    "error": result.get("error"),
                    "raw_response": result.get("raw_response", ""),
                    "text_hash": current_hash,
                    "source": "embedding_similarity_inside_hdbscan_clusters",
                    "source_row_idx_in_semantic_pool": int(source_row_idx),
                    "candidate_idx": int(row.get("candidate_idx", -1)),
                    "best_semantic_similarity": float(row.get("best_semantic_similarity", 0.0)),
                    "best_semantic_rank": int(row.get("best_semantic_rank", -1)),
                    "search_target_labels": row.get("search_target_labels", ""),
                    "cluster_id": int(row.get(CLUSTER_COL)) if pd.notna(row.get(CLUSTER_COL)) else None,
                    "cluster_probability": float(row.get(PROB_COL, 0.0)) if pd.notna(row.get(PROB_COL)) else None,
                    "cluster_title": row.get("cluster_title", ""),
                    "cluster_feature": row.get("cluster_feature", ""),
                    "main_theme": row.get("main_theme", ""),
                    "cluster_matched_labels_str": row.get("matched_labels_str", ""),
                    "cluster_positive_or_negative": row.get("positive_or_negative", ""),
                    "cluster_analysis_confidence": row.get("confidence", 0.0),
                }

                selected_examples.append(example_row)
                append_example_to_disk(example_row)
                existing_hashes.add(current_hash)

                for label in labels_counted:
                    if label in current_total_counts:
                        before = current_total_counts[label]
                        current_total_counts[label] += 1
                        if label in class_bars:
                            # Обновляем progress bar только до TARGET_MIN_COUNT.
                            delta = min(current_total_counts[label], TARGET_MIN_COUNT) - min(before, TARGET_MIN_COUNT)
                            if delta > 0:
                                class_bars[label].update(delta)
                            class_bars[label].set_postfix({
                                "count": current_total_counts[label],
                                "need": max(0, TARGET_MIN_COUNT - current_total_counts[label]),
                            })

                main_bar.set_postfix({
                    "sent": sent_to_model,
                    "saved": len(selected_examples) - saved_examples_before,
                    "skip_no_target": skipped_without_target_label,
                })

            if REQUEST_SLEEP_SECONDS > 0:
                time.sleep(REQUEST_SLEEP_SECONDS)

    finally:
        for bar in class_bars.values():
            bar.close()
        main_bar.close()

    labeled_df = save_full_outputs(selected_examples)
    status_df = build_total_class_status_df(selected_examples)

    stats = {
        "sent_to_model": sent_to_model,
        "scanned_reviews": scanned_reviews,
        "skipped_duplicates": skipped_duplicates,
        "skipped_empty": skipped_empty,
        "skipped_without_target_label": skipped_without_target_label,
        "saved_examples_before": saved_examples_before,
        "saved_examples_after": len(selected_examples),
        "saved_examples_added_this_run": len(selected_examples) - saved_examples_before,
        "target_labels": TARGET_LABELS,
        "target_min_count": TARGET_MIN_COUNT,
        "top_k_per_class": TOP_K_PER_CLASS,
        "top_k_by_label": TOP_K_BY_LABEL,
        "use_prototype_query_expansion": USE_PROTOTYPE_QUERY_EXPANSION,
        "max_examples_as_prototypes_per_label": MAX_EXAMPLES_AS_PROTOTYPES_PER_LABEL,
        "embedding_model_name": EMBEDDING_MODEL_NAME,
        "use_only_candidate_clusters_from_analysis": USE_ONLY_CANDIDATE_CLUSTERS_FROM_ANALYSIS,
        "select_clusters_by_target_rare_labels": SELECT_CLUSTERS_BY_TARGET_RARE_LABELS,
    }
    RUN_STATS_JSON_PATH.write_text(json.dumps(stats, ensure_ascii=False, indent=2), encoding="utf-8")

    print("Готово.")
    print("Отправлено в модель:", sent_to_model)
    print("Сохранено новых строк за этот запуск:", len(selected_examples) - saved_examples_before)
    print("Полный CSV:", OUTPUT_CSV_PATH)
    print("Финальный CSV:", FINAL_CSV_PATH)
    print("Статус классов:", STATUS_CSV_PATH)
    print("Статистика:", RUN_STATS_JSON_PATH)

    return labeled_df, status_df, stats


labeled_df, status_df, stats = collect_extra_examples_from_embedding_candidates()
display(status_df)

Найден текущий результат для продолжения: /content/drive/MyDrive/MLops_project/project/data/labeled/wb_feedbacks_llm_labeled_from_embeddings/extra_from_embedding_similarity_rare_classes.csv
Уже сохранено строк в текущем результате: 45


Подделка / оригинальность:   1%|1         | 1/100 [00:00<?, ?it/s]

Нейтральный / информационный отзыв:   0%|          | 0/100 [00:00<?, ?it/s]

Проблема с размером / посадкой:   2%|2         | 2/100 [00:00<?, ?it/s]

Низкое качество материала:  12%|#2        | 12/100 [00:00<?, ?it/s]

Цена / ценность:   0%|          | 0/100 [00:00<?, ?it/s]

Другая проблема:   0%|          | 0/100 [00:00<?, ?it/s]

Semantic candidates:   0%|          | 0/2891 [00:00<?, ?it/s]


ДОСТИГНУТ ЛИМИТ API / RATE LIMIT
attempt: 1/3
Error code: 429 - {'error': {'message': 'Rate limit reached for model `qwen/qwen3-32b` in organization `org_01ktr8d8t1f29vnk0br9z98mwr` service tier `on_demand` on tokens per day (TPD): Limit 500000, Used 499717, Requested 2827. Please try again in 7m19.603199999s. Need more tokens? Upgrade to Dev Tier today at https://console.groq.com/settings/billing', 'type': 'tokens', 'code': 'rate_limit_exceeded'}}


ДОСТИГНУТ ЛИМИТ API / RATE LIMIT
attempt: 2/3
Error code: 429 - {'error': {'message': 'Rate limit reached for model `qwen/qwen3-32b` in organization `org_01ktr8d8t1f29vnk0br9z98mwr` service tier `on_demand` on tokens per day (TPD): Limit 500000, Used 499704, Requested 2818. Please try again in 7m15.8016s. Need more tokens? Upgrade to Dev Tier today at https://console.groq.com/settings/billing', 'type': 'tokens', 'code': 'rate_limit_exceeded'}}


ДОСТИГНУТ ЛИМИТ API / RATE LIMIT
attempt: 3/3
Error code: 429 - {'error': {'message': 'Rate li

KeyboardInterrupt: 

## 20. Просмотр сохраненного результата

In [ ]:
print("Полный CSV:", OUTPUT_CSV_PATH)
print("Финальный CSV:", FINAL_CSV_PATH)
print("Parquet:", OUTPUT_PARQUET_PATH)
print("Статус классов:", STATUS_CSV_PATH)
print("Статистика:", RUN_STATS_JSON_PATH)

if OUTPUT_CSV_PATH.exists():
    result_df = pd.read_csv(OUTPUT_CSV_PATH)
    print("Сохранено строк:", len(result_df))
    display(result_df.tail(20))
else:
    print("Пока не сохранено ни одного примера.")